# Step 4. Modeling and Comparison

I train six models to predict dropout from the enrollment-time features, and I compare them on the metrics I set in Step 1. I load the split and the preprocessor I saved in Step 3, so I do not re-split or re-prepare anything.

The plan for this notebook is simple.

1. Load the saved split and pipeline.
2. Decide how I judge a model, and how I handle the uneven split.
3. Train six models, tuned inside cross-validation on the training set.
4. Score the test set once, compare, and pick one model with reasons.

In [ ]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

import numpy as np
import pandas as pd
import joblib
from src.paths import ROOT

proc = ROOT / "data" / "processed"
if not (proc / "train.csv").exists():
    raise FileNotFoundError(
        "Run Step 3 first. It saves the split to data/processed and the preprocessor to models."
    )

train = pd.read_csv(proc / "train.csv")
test = pd.read_csv(proc / "test.csv")
ytr = train.pop("dropout")
yte = test.pop("dropout")
Xtr, Xte = train, test
preprocessor = joblib.load(ROOT / "models" / "preprocessor.joblib")
print("train", Xtr.shape, "| test", Xte.shape)

## How I Judge the Models

The model gives each student a risk score from 0 to 1, not a plain yes or no. I turn that into a decision with a cutoff, and the cutoff can move. The measures below score the model across every cutoff at once, so no single choice drives the result. AUC means area under the curve.

I judge on three things, in this order.

Recall on dropout. Of the students who truly drop out, how many did the model catch? A missed student is the costly error, since that student loses the help. This is my first priority.

PR AUC, from precision and recall. Precision asks, of the students I flag, how many truly drop out. PR AUC pairs precision with recall across every cutoff. It ignores the graduates I correctly leave alone, so it stays honest on the uneven 39 to 61 split.

ROC AUC. I take one random dropout and one random graduate. This is the chance the model gives the dropout the higher score. It reads as a ranking score, 1.0 for perfect order and 0.5 for a coin flip. It matches the real use, a ranked list staff work down.

I report F1 too, but I do not lead with it, since it depends on one fixed cutoff.

## How I Handle the Uneven Split

The dropout rate is 39 percent, so the split is mildly uneven, not severe. I use class weights, which tell each model to treat a dropout as more costly than a graduate, in that same proportion. This is the light touch that fits a mild imbalance. I skip SMOTE, which invents synthetic rows and is better saved for a heavier imbalance. The one exception is the neural net, which has no class weight setting, so it trains unweighted. The mild imbalance makes that acceptable.

## The Six Models

I train six models, from simple to flexible.

Logistic regression. A straight line boundary. Simple, fast, and easy to read, since each feature gets a weight I can inspect.

Decision tree. A set of yes or no splits. Easy to follow, but it overfits on its own.

Random forest. Many trees averaged. The averaging cuts the overfitting of a single tree.

XGBoost. Trees built in sequence, each one fixing the last one's mistakes. Strong on tabular data.

SVM. A boundary that aims for the widest gap between the two groups. Stable, but slower to train.

MLP, a small neural net. The deep learning entry, added as a test.

## Why I Include a Neural Net

I add the MLP to test one question. Does deep learning beat trees on data this size and shape?

Neural nets shine on large, high dimensional data with rich structure, like images, audio, and text, where they learn patterns simpler models cannot. They need many rows to do that well, and they cost more to train and tune.

This dataset is small and tabular, 2904 training rows and a mix of numbers and categories. On data like this, gradient boosted trees usually win, since they handle mixed types and small samples well without a deep network. I expect the MLP to lose to XGBoost and the forest here. That result is worth showing, not hiding, since it tells me deep learning is the wrong tool for this problem. A finding, not a failure.

## Cross-Validation, Then One Test Score

A single train and test split can be lucky or unlucky. To trust a model, I check it on five folds of the training data, using stratified five-fold cross-validation, which keeps the 39 percent dropout rate in each fold. I tune each model inside this cross-validation, so the tuning never sees the test set.

I score the test set once, at the end, after the models are chosen and tuned. The test set is the final check, not a dial I turn.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, roc_auc_score, average_precision_score, f1_score

# weight of the positive class, so XGBoost treats dropout as more costly
spw = (ytr == 0).sum() / (ytr == 1).sum()
cv = StratifiedKFold(5, shuffle=True, random_state=42)

model_grid = {
    "logistic regression": (LogisticRegression(max_iter=2000, class_weight="balanced"), {"m__C": [0.1, 1, 10]}),
    "decision tree": (DecisionTreeClassifier(class_weight="balanced", random_state=42), {"m__max_depth": [4, 8, None]}),
    "random forest": (RandomForestClassifier(class_weight="balanced", n_estimators=300, random_state=42, n_jobs=-1), {"m__max_depth": [None, 12], "m__min_samples_leaf": [1, 5]}),
    "xgboost": (XGBClassifier(scale_pos_weight=spw, n_estimators=300, eval_metric="logloss", random_state=42, n_jobs=-1), {"m__max_depth": [3, 5], "m__learning_rate": [0.05, 0.1]}),
    "svm": (SVC(class_weight="balanced", probability=True, random_state=42), {"m__C": [1, 10]}),
    "mlp (neural net)": (MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=600, random_state=42), {"m__alpha": [0.001, 0.01]}),
}

# I tune each model inside 5-fold cross-validation on the training set,
# then score the held-out test set once.
results, fitted = [], {}
for name, (est, grid) in model_grid.items():
    pipe = Pipeline([("pre", clone(preprocessor)), ("m", est)])
    search = GridSearchCV(pipe, grid, scoring="average_precision", cv=cv, n_jobs=-1)
    search.fit(Xtr, ytr)
    proba = search.predict_proba(Xte)[:, 1]
    pred = search.predict(Xte)
    results.append({
        "model": name,
        "cv PR-AUC": round(search.best_score_, 3),
        "PR-AUC": round(average_precision_score(yte, proba), 3),
        "ROC-AUC": round(roc_auc_score(yte, proba), 3),
        "recall": round(recall_score(yte, pred), 3),
        "F1": round(f1_score(yte, pred), 3),
    })
    fitted[name] = search.best_estimator_

comparison = pd.DataFrame(results).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
comparison

## What the Comparison Shows

Four models cluster at the top on PR AUC. SVM and random forest tie at 0.831, XGBoost sits at 0.829, and logistic regression is just behind at 0.822. The gap between them is small.

Logistic regression catches the most dropouts, with a recall of 0.775, ahead of every other model. Since catching at risk students is my first priority, that matters.

The neural net loses. Its PR AUC of 0.776 sits below every model except the lone decision tree. This answers the question I set. Deep learning does not beat trees, or even a straight line, on data this small and tabular.

The single decision tree is weakest, at 0.716, which is expected. It overfits on its own, and the forest and the boosting fix that by combining many trees.

## The Model I Choose

I choose logistic regression.

It catches the most dropouts, with the best recall at 0.775, which is the metric I care about most. Its PR AUC of 0.822 is within 0.01 of the top, and its ROC AUC of 0.855 is close behind the best. So it gives up almost nothing on ranking while catching more at risk students.

It is also the simplest and the easiest to read. Each feature carries a weight I can point to, which helps the fairness audit in Step 5 and the plain explanation for staff. The heavier models buy a tiny gain in PR AUC at the cost of that clarity, and here the clarity is worth more.

In [ ]:
import json

chosen = "logistic regression"
joblib.dump(fitted[chosen], ROOT / "models" / "model.joblib")
with open(ROOT / "models" / "metrics.json", "w") as f:
    json.dump({"chosen": chosen, "results": results}, f, indent=2)
print("saved model.joblib (", chosen, ") and metrics.json")

## What Step 4 Settles

1. I trained and tuned six models on the enrollment-time features, using cross-validation on the training set.
2. The top four are close on PR AUC, and logistic regression catches the most dropouts.
3. The neural net lost to the simpler models, which answers my deep learning question for this data.
4. I chose logistic regression, for its recall, its near-top ranking, and its clarity.
5. I saved the chosen model and the full comparison to models.

Step 5 takes this model, explains its decisions, checks it for bias across the four sensitive attributes, and tries to reduce any unfairness.